### DSAI3204 IoT Application Development: Project

#### Maryam Mahaboob
#### Sensors: VOC, temperature & humidity (Arduino MKR IoT Carrier)

## Summary

**Goal:** a Bayesian neural network (Pyro) that predicts next-minute humidity with uncertainty bands; plus a deeper Bayesian network for VOC (negative result).

**Data:** the three raw files in `../data/raw/`, resampled to 1-minute means and merged in this notebook (21,016 rows). The saved outputs come from the original run on the same raw CSVs.

**Split:** random 80/20 `train_test_split` (shuffled); the VOC model uses shuffled 5-fold cross-validation. **Horizon:** 1 minute.

**Key result (saved output, standardized units) vs baseline:**

| | R² | MSE | MAE |
|---|---|---|---|
| BNN (3 → 10 → 1), humidity at t+1 | 0.9994 | 0.0006 | 0.0165 |
| Baseline: repeat the last value | 0.9994 | 0.00065 | 0.0163 |

The baseline was computed later, during a review of the project. **The BNN scores the same as simply repeating the last humidity value**, so it shows no forecasting skill. What it does show is how to get credible intervals from a Bayesian model.

VOC with the deeper BNN (64-32-16, lag and rolling features): R² −1.83, −1.61 and −6.91 on the first three folds (worse than predicting the mean). Training was unstable; fold 4 did not finish.

**Caveats**
- Shuffled split and shuffled K-fold on a time series: neighbouring minutes end up in both train and test.
- The StandardScaler is fit on all rows before the split (the VOC model does it correctly inside each fold).
- BNN = persistence (see above). Do not quote R² 0.9994 as a forecasting result.
- Changes during cleanup: one missing line was added in the time-conversion cell (humidity), and the cells after the first BNN were removed (a 60-minute recursive forecast, a full re-run, a shuffled 5-fold run for humidity, and three cells that ended in errors). The VOC experiment is kept at the end.


##### CONTENTS: voc, temperature & humidity Predictions & Forecasting Attempts - Trial & Error Pre-Processing Code

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
import torch
from pyro.infer import SVI, Predictive, Trace_ELBO
from pyro.infer.autoguide import AutoNormal
from pyro.nn import PyroModule, PyroSample
from pyro.optim import Adam, ClippedAdam
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

SEED = 42  # fixed seed for every training run and every posterior sample
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
humidity = pd.read_csv("../data/raw/humidity_raw.csv.gz")
temp = pd.read_csv("../data/raw/temperature_raw.csv.gz")
voc = pd.read_csv("../data/raw/voc_raw.csv.gz")

humidity.info()

In [ ]:
temp["time"] = pd.to_datetime(temp["time"], format="ISO8601", utc=True)
voc["time"] = pd.to_datetime(voc["time"], format="ISO8601", utc=True)
temp.set_index('time', inplace=True)
voc.set_index('time', inplace=True)

humidity["time"] = pd.to_datetime(humidity["time"], format="ISO8601", utc=True)
humidity.set_index('time', inplace=True)

In [ ]:
# Now resample to 1-minute intervals
humidity_minutely = humidity.resample('1min').mean()
temp_minutely = temp.resample("1min").mean()
voc_minutely = voc.resample("1min").mean()

In [ ]:
#shapes of the data 

print(humidity_minutely.info())
print(temp_minutely.info())
print(voc_minutely.info())


In [ ]:
## renaming 

humidity_minutely.rename(columns={'value':'humidity'},inplace=True)
temp_minutely.rename(columns={'value':'temperature'},inplace=True)
voc_minutely.rename(columns={"value":"VOC"},inplace=True)


In [ ]:
##concat all of them 
data = pd.concat([humidity_minutely, temp_minutely, voc_minutely], axis=1)
data.head(10)

In [ ]:
data.dropna(inplace=True)

In [ ]:
# --- Chronological folds, shared by the humidity model and the VOC model ---
# TimeSeriesSplit gives 5 expanding-window folds; every training minute comes before every test minute.
# The fold boundaries are stored as timestamps so the VOC model below uses exactly the same folds.
tscv = TimeSeriesSplit(n_splits=5)
FOLDS = []
for tr_idx, te_idx in tscv.split(data.index[:-1]):
    FOLDS.append((data.index[tr_idx[-1]], data.index[te_idx[0]], data.index[te_idx[-1]]))
for k, (tr_end, te_start, te_end) in enumerate(FOLDS, 1):
    print(f"Fold {k}: train up to {tr_end}, test {te_start} to {te_end}")

# Humidity model: features are humidity, temperature and VOC at minute t; target is humidity at t+1
X_all = data[['humidity', 'temperature', 'VOC']].iloc[:-1]
y_all = data['humidity'].shift(-1).iloc[:-1]
print(f"Rows: {len(X_all)}")

In [ ]:
class BayesianRegression(PyroModule):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = PyroModule[torch.nn.Linear](in_features, 10)
        self.fc1.weight = PyroSample(dist.Normal(0., 1.).expand([10, in_features]).to_event(2))
        self.fc1.bias = PyroSample(dist.Normal(0., 1.).expand([10]).to_event(1))

        self.out = PyroModule[torch.nn.Linear](10, 1)
        self.out.weight = PyroSample(dist.Normal(0., 1.).expand([1, 10]).to_event(2))
        self.out.bias = PyroSample(dist.Normal(0., 1.).expand([1]).to_event(1))

    def forward(self, x, y=None):
        x = torch.relu(self.fc1(x))
        mean = self.out(x).squeeze(-1)
        sigma = pyro.sample("sigma", dist.Uniform(0., 1.))
        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)
        return mean


def scores(y_true, y_pred, lower=None, upper=None):
    out = {"R2": r2_score(y_true, y_pred), "RMSE": root_mean_squared_error(y_true, y_pred),
           "MAE": mean_absolute_error(y_true, y_pred)}
    out["coverage_95"] = np.mean((y_true >= lower) & (y_true <= upper)) if lower is not None else np.nan
    return out

In [ ]:
# Train and score the humidity BNN on each chronological fold (scaler fitted on the training rows only)
hum_rows = []
for k, (tr_end, te_start, te_end) in enumerate(FOLDS, 1):
    X_tr, y_tr = X_all.loc[:tr_end], y_all.loc[:tr_end]
    X_te, y_te = X_all.loc[te_start:te_end], y_all.loc[te_start:te_end]

    x_scaler = StandardScaler().fit(X_tr)
    y_mean, y_std = y_tr.mean(), y_tr.std()
    X_train = torch.tensor(x_scaler.transform(X_tr), dtype=torch.float32)
    y_train = torch.tensor(((y_tr - y_mean) / y_std).to_numpy(), dtype=torch.float32)
    X_test = torch.tensor(x_scaler.transform(X_te), dtype=torch.float32)

    pyro.set_rng_seed(SEED)
    pyro.clear_param_store()
    model = BayesianRegression(in_features=X_train.shape[1])
    guide = AutoNormal(model)
    svi = SVI(model, guide, Adam({"lr": 0.01}), loss=Trace_ELBO())

    # Training loop
    num_iterations = 2000
    print(f"\n--- Humidity fold {k} ---")
    for i in range(num_iterations):
        loss = svi.step(X_train, y_train)
        if i % 500 == 0:
            print(f"[{i}] Loss: {loss:.4f}")

    pyro.set_rng_seed(SEED)
    samples = Predictive(model, guide=guide, num_samples=1000)(X_test)["obs"].detach().numpy()
    samples = samples * y_std + y_mean          # back to %RH
    pred_mean = samples.mean(axis=0)
    lower, upper = np.percentile(samples, [2.5, 97.5], axis=0)
    y_true = y_te.to_numpy()

    bnn = scores(y_true, pred_mean, lower, upper)
    persist = scores(y_true, X_te['humidity'].to_numpy())
    print(f"BNN:         R2 {bnn['R2']:.4f}  RMSE {bnn['RMSE']:.4f}  MAE {bnn['MAE']:.4f}  95% interval coverage {bnn['coverage_95']:.3f}")
    print(f"Persistence: R2 {persist['R2']:.4f}  RMSE {persist['RMSE']:.4f}  MAE {persist['MAE']:.4f}")
    hum_rows.append({"fold": k, **{f"BNN {m}": v for m, v in bnn.items()},
                     **{f"persistence {m}": v for m, v in persist.items() if m != "coverage_95"}})

hum_results = pd.DataFrame(hum_rows).set_index("fold")

In [ ]:
# Prediction with uncertainty on the last fold, plotted over time (one day for readability)
window = slice(-1440, None)
plt.figure(figsize=(12, 5))
plt.plot(y_te.index[window], pred_mean[window], label="Predicted (BNN mean)")
plt.fill_between(y_te.index[window], lower[window], upper[window], alpha=0.3, label="95% interval")
plt.plot(y_te.index[window], y_true[window], label="True", alpha=0.7)
plt.legend()
plt.title("BNN next-minute humidity with uncertainty (last day of fold 5)")
plt.ylabel("Humidity (%RH)")
plt.tight_layout()
plt.savefig(FIG_DIR / "bnn_humidity_uncertainty.png", bbox_inches="tight")
plt.show()

In [ ]:
# Humidity BNN vs persistence on the same chronological folds (%RH)
print(hum_results.round(4).to_string())
print("\nMean over folds:")
print(hum_results.mean().round(4).to_string())

In [ ]:
# --- Data prep with lags & rolling stats ---
def prepare_features(data):
    data = data.copy()
    # Add lag features
    for lag in range(1, 6):
        data[f'humidity_lag_{lag}'] = data['humidity'].shift(lag)
        data[f'temperature_lag_{lag}'] = data['temperature'].shift(lag)
        data[f'VOC_lag_{lag}'] = data['VOC'].shift(lag)

    # Rolling stats with window=5 over the 5 minutes before t (shift(1)), so no feature sees minute t
    for col in ['humidity', 'temperature', 'VOC']:
        data[f'{col}_roll_mean'] = data[col].shift(1).rolling(window=5).mean()
        data[f'{col}_roll_std'] = data[col].shift(1).rolling(window=5).std()

    # Drop rows with NaNs from lagging/rolling
    data.dropna(inplace=True)

    # Cyclic time features from datetime index (known in advance)
    hours = data.index.hour + data.index.minute / 60
    data['time_sin'] = np.sin(2 * np.pi * hours / 24)
    data['time_cos'] = np.cos(2 * np.pi * hours / 24)

    return data

# --- Bayesian regression model with 3 hidden layers ---
class DeepBayesianRegression(PyroModule):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = PyroModule[torch.nn.Linear](in_features, 64)
        self.fc1.weight = PyroSample(dist.Normal(0., 1.).expand([64, in_features]).to_event(2))
        self.fc1.bias = PyroSample(dist.Normal(0., 1.).expand([64]).to_event(1))

        self.fc2 = PyroModule[torch.nn.Linear](64, 32)
        self.fc2.weight = PyroSample(dist.Normal(0., 1.).expand([32, 64]).to_event(2))
        self.fc2.bias = PyroSample(dist.Normal(0., 1.).expand([32]).to_event(1))

        self.fc3 = PyroModule[torch.nn.Linear](32, 16)
        self.fc3.weight = PyroSample(dist.Normal(0., 1.).expand([16, 32]).to_event(2))
        self.fc3.bias = PyroSample(dist.Normal(0., 1.).expand([16]).to_event(1))

        self.out = PyroModule[torch.nn.Linear](16, 1)
        self.out.weight = PyroSample(dist.Normal(0., 1.).expand([1, 16]).to_event(2))
        self.out.bias = PyroSample(dist.Normal(0., 1.).expand([1]).to_event(1))

    def forward(self, x, y=None):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        mean = self.out(x).squeeze(-1)
        sigma = pyro.sample("sigma", dist.Uniform(0., 1.))
        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)
        return mean

# --- Training with LR scheduling ---
def train_bayesian_regression(X_train, y_train, num_steps=3000, lr=0.01, log_interval=100):
    pyro.set_rng_seed(SEED)
    pyro.clear_param_store()
    model = DeepBayesianRegression(in_features=X_train.shape[1])
    guide = AutoNormal(model)
    optimizer = ClippedAdam({"lr": lr})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32)

    losses = []
    for step in range(num_steps):
        loss = svi.step(X_t, y_t)
        if step % log_interval == 0:
            losses.append(loss)
        if step % 1000 == 0:
            print(f"Step {step} : loss = {loss:.1f}")
    return model, guide, losses

# --- Prediction with uncertainty ---
def predict_with_uncertainty(model, guide, X, num_samples=1000):
    pyro.set_rng_seed(SEED)
    predictive = Predictive(model, guide=guide, num_samples=num_samples)
    X_t = torch.tensor(X, dtype=torch.float32)
    samples = predictive(X_t)
    obs_samples = samples["obs"].detach().numpy()
    return obs_samples

# === Main ===
voc_data = prepare_features(data)

# Features: past values only (lags, rolling stats over t-5 ... t-1) plus the time of day.
# Humidity and temperature at minute t are not used (they would not be known yet).
feature_cols = [
    'time_sin', 'time_cos',
    *[f'humidity_lag_{i}' for i in range(1,6)],
    *[f'temperature_lag_{i}' for i in range(1,6)],
    *[f'VOC_lag_{i}' for i in range(1,6)],
    'humidity_roll_mean', 'humidity_roll_std',
    'temperature_roll_mean', 'temperature_roll_std',
    'VOC_roll_mean', 'VOC_roll_std',
]
target_col = 'VOC'

# Same chronological folds as the humidity model (fold boundaries as timestamps)
voc_rows = []
for fold, (tr_end, te_start, te_end) in enumerate(FOLDS, 1):
    print(f"\n--- Fold {fold} ---")
    tr = voc_data.loc[:tr_end]
    te = voc_data.loc[te_start:te_end]

    x_scaler = StandardScaler().fit(tr[feature_cols])
    X_train = x_scaler.transform(tr[feature_cols])
    X_test = x_scaler.transform(te[feature_cols])
    # Target standardised with training statistics only
    y_mean, y_std = tr[target_col].mean(), tr[target_col].std()
    y_train = ((tr[target_col] - y_mean) / y_std).to_numpy()
    y_test = te[target_col].to_numpy()

    model, guide, losses = train_bayesian_regression(X_train, y_train, num_steps=3000, lr=0.01)

    # Plot training loss
    plt.figure(figsize=(8,4))
    plt.plot(np.arange(0, 3000, 100), losses)
    plt.title(f"Training Loss (ELBO) - Fold {fold}")
    plt.xlabel("Training Step")
    plt.ylabel("ELBO Loss")
    plt.grid(True)
    plt.show()

    obs = predict_with_uncertainty(model, guide, X_test) * y_std + y_mean
    mean_preds = obs.mean(axis=0)
    lower, upper = np.percentile(obs, [2.5, 97.5], axis=0)

    bnn = scores(y_test, mean_preds, lower, upper)
    persist = scores(y_test, te['VOC_lag_1'].to_numpy())
    print(f"BNN:         R2 {bnn['R2']:.4f}  RMSE {bnn['RMSE']:.4f}  MAE {bnn['MAE']:.4f}  95% interval coverage {bnn['coverage_95']:.3f}")
    print(f"Persistence: R2 {persist['R2']:.4f}  RMSE {persist['RMSE']:.4f}  MAE {persist['MAE']:.4f}")
    voc_rows.append({"fold": fold, **{f"BNN {m}": v for m, v in bnn.items()},
                     **{f"persistence {m}": v for m, v in persist.items() if m != "coverage_95"}})

    # Plot predictions with credible intervals (one day of the test fold)
    window = slice(-1440, None)
    plt.figure(figsize=(10,6))
    plt.plot(te.index[window], y_test[window], label="Actual VOC", marker='o', markersize=2, linestyle='None', alpha=0.6)
    plt.plot(te.index[window], mean_preds[window], label="Predicted VOC")
    plt.fill_between(te.index[window], lower[window], upper[window], color='gray', alpha=0.3, label="95% Credible Interval")
    plt.title(f"VOC Prediction with Uncertainty - Fold {fold} (last day)")
    plt.legend()
    if fold == len(FOLDS):
        plt.savefig(FIG_DIR / "bnn_voc_uncertainty.png", bbox_inches="tight")
    plt.show()

voc_results = pd.DataFrame(voc_rows).set_index("fold")
print("\nVOC BNN vs persistence on the same chronological folds (ppm-equivalent):")
print(voc_results.round(4).to_string())
print("\nMean over folds:")
print(voc_results.mean().round(4).to_string())

In [ ]:
# --- Leakage checks ---
for k, (tr_end, te_start, te_end) in enumerate(FOLDS, 1):
    print(f"Fold {k}: last train {tr_end} < first test {te_start}: {tr_end < te_start}")
print("Scalers and target statistics are fitted inside each fold on the training rows only.")
print("Humidity model: inputs at minute t, target at t+1.")
voc_feature_check = voc_data[['VOC_roll_mean']].join(data['VOC'].shift(1).rolling(5).mean().rename('expected'))
print("VOC model: rolling features end at t-1:", bool(np.allclose(voc_feature_check['VOC_roll_mean'], voc_feature_check['expected'])))
print("VOC model: no same-minute humidity, temperature or VOC column in the features:",
      not set(feature_cols) & {'humidity', 'temperature', 'VOC'})